Developing an Intent Classification System using Machine Learning that can automatically detect user intent from text input using machine learning techniques.<br>
👉 Example:<br>
"Hi" → Greeting<br>
"Order pizza" → Food Order<br>
"What’s the weather?" → Weather Query<br>
This is a core component of chatbots, virtual assistants, and NLP systems.


In [ ]:
# ============================================================
# CELL 1: Importing Libraries and Setting Up the Environment
# ============================================================

# Import built-in Python libraries
import os
import re
import string
import joblib
import warnings

# Import data handling libraries
import numpy as np
import pandas as pd

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Import machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier

# Import evaluation metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Ignore unnecessary warnings
warnings.filterwarnings("ignore")

# Print confirmation message
print("Project setup completed successfully!")

In [ ]:
# Function to load and explore the dataset
def load_dataset(file_path):
    """
    Load and perform initial exploration of the dataset
    """
    # upload the dataset
    df = pd.read_csv(file_path)
    
    #check the first few rows of the dataset
    print("First 5 rows of dataset:")
    print(df.head())
    
    # check the label distribution
    print("\nLabel Distribution:")
    print(df['label'].value_counts())
    
    # plot the label distribution with default colors
    plt.figure(figsize=(8, 5))
    sns.countplot(x='label', data=df)
    plt.title('Label Distribution in Dataset', fontsize=14, fontweight='bold')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('figures/label_distribution.png')
    plt.show()
    
    return df

# Load dataset
df = load_dataset('news_dataset.csv')

In [ ]:
# perform data cleaning before preprocessing and training the models
def clean_dataset(df):
    """
    Clean the dataset by handling missing values, duplicates, and filtering reviews
    """
    # checking for missing values
    print("Missing Values:")
    print(df.isnull().sum())
    
    # since we got some missing values in the Text column, we will drop those rows
    df.dropna(subset=['Text'], inplace=True)
    
    #again check for missing values
    print("\nMissing Values After Cleaning:")
    print(df.isnull().sum())
    
    #checking for duplicates
    print(f"\nDuplicate rows found: {df.duplicated().sum()}")
    
    #we also got some duplicate rows, we will drop those as well
    df.drop_duplicates(inplace=True)
    
    #again check for duplicates
    print(f"Duplicates after removal: {df.duplicated().sum()}")
    
    #checking the head of the dataset after preprocessing
    print("\nFirst 5 rows after cleaning:")
    print(df.head())
    
    # Visualize Text length distribution after cleaning
    plt.figure(figsize=(10, 5))
    plt.hist(df['Text'].apply(lambda x: len(str(x).split())), bins=30, edgecolor='black', alpha=0.7)
    plt.axvline(x=20, color='red', linestyle='--', linewidth=2, label='Min (20 words)')
    plt.title('Distribution of Text Word Counts After Cleaning', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Words')
    plt.ylabel('Frequency')
    plt.legend()
    plt.tight_layout()
    plt.savefig('figures/Text_length_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return df

# Clean dataset (before preprocessing)
df = clean_dataset(df)

In [ ]:
# preprocess the text data using advance text cleaning techniques to improve the performance of the models
# Load stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Advanced NLP preprocessing pipeline:
    - Lowercasing
    - Remove URLs and HTML tags
    - Normalize currency and numbers
    - Remove special characters
    - Tokenization (NLTK)
    - POS tagging
    - Lemmatization with POS
    - Stopword removal
    - Remove short tokens
    - Clean extra spaces
    """
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    
    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    
    # 4. Normalize currency (e.g., $100.50 → money)
    text = re.sub(r'\$\s?\d+(\.\d+)?', ' money ', text)
    
    # 5. Normalize numbers (e.g., 123 → number)
    text = re.sub(r'\d+(\.\d+)?', ' number ', text)
    
    # 6. Remove special characters & punctuation
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 7. Tokenization using NLTK
    tokens = word_tokenize(text)
    
    # 8. POS tagging
    pos_tags = pos_tag(tokens)
    
    # 9. Helper function for POS conversion (inside function as requested)
    def get_wordnet_pos(tag):
        if tag.startswith('J'):
            return 'a'  # adjective
        elif tag.startswith('V'):
            return 'v'  # verb
        elif tag.startswith('N'):
            return 'n'  # noun
        elif tag.startswith('R'):
            return 'r'  # adverb
        else:
            return 'n'
    
    # 10. Lemmatization + stopword removal + short word filtering
    cleaned_words = []
    for word, tag in pos_tags:
        if word not in stop_words and len(word) > 2:
            pos = get_wordnet_pos(tag)
            lemma = lemmatizer.lemmatize(word, pos)
            cleaned_words.append(lemma)
    
    # 11. Join tokens
    text = ' '.join(cleaned_words)
    
    # 12. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# apply the preprocessing function to the reviews
print("Applying text preprocessing...")
df['Text'] = df['Text'].apply(preprocess_text)

# Remove any empty strings after preprocessing
df = df[df['Text'] != ""]

# check the length and sample of the dataset after preprocessing
print("Text preprocessing completed!")
print(f"Number of reviews after preprocessing: {len(df)}")
print("\nSample preprocessed text:")
print(df['Text'].iloc[0][:200] + "...")

In [ ]:
# Prepare features using TF-IDF and labels using LabelEncoder
def prepare_features(df):
    """
    Prepare features using TF-IDF and labels using LabelEncoder
    """
    # now we will split the dataset into features and target variable
    X = df['Text']  # Features (Text)
    y = df['label']  # Target variable (label)
    
    # we will convert the target variable into numerical format using LabelEncoder and save the encoder for later use
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    # lets see the mapping of the labels
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    print("Label Mapping:", label_mapping)
    
    # checking the shape of the features and target variable
    print("Features shape:", X.shape)
    print("Target variable shape:", y_encoded.shape)
    
    # train test split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
    
    print(f"\nTraining set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")
    
    return X_train, X_test, y_train, y_test, label_encoder

# Prepare features
X_train, X_test, y_train, y_test, label_encoder = prepare_features(df)

In [ ]:
# Baseline model function that trains a Logistic Regression model, evaluates it, and plots the confusion matrix

def baseline_model(X_train, y_train, X_test, y_test):
    print("\n" + "="*60)
    print(" BASELINE MODEL")
    print("="*60)
    
    baseline_pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000)),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    
    baseline_pipe.fit(X_train, y_train)
    baseline_preds = baseline_pipe.predict(X_test)
    baseline_acc = accuracy_score(y_test, baseline_preds)
    
    print(f"Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_test, baseline_preds, target_names=['Fake', 'Real']))
    
    # Confusion Matrix for Baseline Model
    plt.figure(figsize=(6,4))
    sns.heatmap(confusion_matrix(y_test, baseline_preds), annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix - Baseline Model")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig('figures/confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return baseline_pipe, baseline_acc, baseline_preds

baseline_model, baseline_acc, baseline_preds = baseline_model(X_train, y_train, X_test, y_test)